# SSY098
## Project 3: Structure-from-Motion in 2D

In this project, you will implement a Structure-from-Motion pipeline in 2D, which is equivalent
to stitching together multiple 2D images of a planar surface.

### AI Usage Statement
<!-- Claude (claude-sonnet-4-6 via Claude Code) was used as an assistant during the completion of this lab. The AI was not used to directly write or generate implementation code, but was consulted for conceptual explanations and debugging guidance throughout the assignment.                          
                       
Specifically, the AI was used to:
  - Explain inverse warping and why target pixel locations are always known independently of the warp
  - Explain affine transformation estimation using np.linalg.lstsq, including why the point matrix must be augmented with ones to solve for translation simultaneously with the linear part        
  - Clarify NumPy concepts such as np.squeeze vs slicing for removing array elements, axis parameter behavior in np.sum, boolean array indexing with ~, and .sum() on boolean arrays                   
  - Debug a cv.warpPerspective dtype error requiring float64 transformation matrices                 
  - Debug an incorrect transformation matrix construction using np.ones instead of np.zeros, causing a corrupted perspective warp
  - Debug a cv.drawMatchesKnn argument error caused by passing coordinate arrays instead of cv.KeyPoint objects
  - Explain RANSAC, including the role of the minimal solver
  - Explain histogram equalization and its role in making SIFT features more comparable across images with different contrast distributions
  - Explain deformable B-spline registration, shrink factors and the coarse-to-fine multi-resolution scheme
                                                                                                     
All implementations were written by me, with the AI serving as an explanatory resource and assistant during debugging. -->

 ## Imports
 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import pandas as pd

## Homography Estimation With RANSAC

In [ ]:
def estimate_affine(pts: np.ndarray, pts_prime: np.ndarray):
    """
    Estimate the Homography H

    Args:
    - pts (np.ndarray): Original points, shape (2, 4)
    - pts_prime (np.ndarray): Transformed points, shape (2, 4)
    
    Returns:
    ?
    """

In [ ]:
def ransac_fit_homography(pts: np.ndarray, pts_prime: np.ndarray, thresh: float, n_iter: int = 10000, max_inliers: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Estimate the affine transformation matrix A and translation vector t
    using RANSAC to handle outliers.
    
    Args:
    - pts (np.ndarray): Original points, shape (2, n)
    - pts_prime (np.ndarray): Transformed points, shape (2, n)
    - thresh (float): Threshold for considering a point as an inlier
    - n_iter (int): Number of iterations for RANSAC
    - max_inliers (int): Maximum number of inliers found

    Returns:
    ?
    """
    max_inliers = 0
    for _ in range(n_iter):                                           
        idx = np.random.choice(pts.shape[1], 3, replace=False)                                       
        A, t = estimate_affine(pts[:, idx], pts_prime[:, idx])
        
        rss = residual_sum_of_squares(A, t, pts, pts_prime)                      
        inliers = rss < thresh
        
        if inliers.sum() > max_inliers:                                                              
            max_inliers = inliers.sum()
            best_A, best_t = A, t

    return best_A, best_t

## BB